In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=768zeukEAFoN2NWKWZt2aycQr2uEdI&access_type=offline&code_challenge=qu3B4CLwG4_Abo4mhFg2gs5zi9Yo8wp_MFvEZvQ4pJg&code_challenge_method=S256


Credentials saved to file: [/Users/meghakaladharreddypothamsetty/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "zprocure" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


Updates are available for some G

In [2]:
import asyncio
from google import genai
from google.genai import types
import os
# ---------- Setup ----------

client = genai.Client(
    vertexai=True,
    project="aistimate",
    location="global",
)

model_name = "gemini-2.5-pro"

generate_content_config = types.GenerateContentConfig(
    temperature=0,
    top_p=1,
    seed=7,
    max_output_tokens=65535,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ],
    thinking_config=types.ThinkingConfig(thinking_budget=-1),
)

# ---------- Helper ----------

def make_part(path: str) -> types.Part:
    with open(path, "rb") as f:
        data = f.read()
    ext = path.split(".")[-1].lower()
    mime = {
        "pdf": "application/pdf",
        "png": "image/png",
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "txt": "text/plain",
        "json": "application/json"
    }.get(ext, "application/octet-stream")
    return types.Part.from_bytes(data=data, mime_type=mime)
# ---------- Output Saving ----------







In [3]:
def build_prompts():
    return {
        "CODE_LOOKUP": """You are a building code compliance analyst preparing an expert report from the attached carrier estimate or supplemental documents.

OBJECTIVE:
Extract location-specific property information and generate a code compliance matrix using standardized methodology to ensure consistent analysis across multiple runs.

PART 1: PROPERTY IDENTIFICATION

Extract and display the following details explicitly from carrier estimate documents:

| Field | Value | Source Citation |
|-------|-------|-----------------|
| Full Street Address | [Complete address] | [Document page/section] |
| Municipality/Jurisdiction | [City name] | [Carrier estimate header] |
| County | [County name] | [Address verification] |
| ZIP Code | [5-digit code] | [Carrier estimate] |
| Inspection/Report Date | [MM/DD/YYYY] | [Document date] |
| Carrier Estimate Date | [MM/DD/YYYY] | [Estimate generation date] |
| Initial Carrier Total | [$XX,XXX.XX] | [Final total from carrier estimate] |
| Price List Code | [Regional code] | [Carrier estimate] |

MANDATORY: All fields must include source citations.

TAX RATE DETERMINATION:
Using the property ZIP code, determine jurisdiction-specific tax rates:

TAX RATE PROTOCOL:
1. Extract ZIP code from property address
2. Determine county and municipality from ZIP
3. Research current sales tax rate for construction materials
4. Apply rate only to material portion of line items
5. Document rate source and effective date

Example: "8.25% (Source: Texas Comptroller - Johnson County rate effective 2024)"

PART 2: CODE STACK DETERMINATION

Using the jurisdiction and inspection date, determine the full set of codes applicable at the time of inspection:

| Code Type | Version/Edition | Citation Source | Applied Amendments | Enforceability Level |
|-----------|------------------|------------------|---------------------|------------------------|
| IRC | [Year] IRC | [ICC database, city site] | [NCTCOG mods] | Mandatory |
| NEC | [Year] NEC | [NEC.gov] | [none] | Mandatory |
| IECC | [Year] IECC | [DOE/state site] | [state energy code mods] | Mandatory |

For each code:
- Confirm it applies to residential work
- Cross-verify adoption at municipal, county, and state levels
- Include any stricter local amendments or enforcement practices

PART 3: CODE COMPLIANCE MATRIX

Build a matrix of building components and related code requirements:

| Affected System | Code Section | Code Summary | Interpretation | Required By Code | Present in Carrier Estimate | Justification |
|------------------|---------------|----------------|------------------|-------------------|-------------------------------|----------------|
| Roofing | IRC R905.2.8.5 | Drip edge required at eaves & rakes | Must be installed per manufacturer & code | Yes | No | Missing on eaves; required due to full shingle tear-off |

Mandatory Coverage Areas (review all):
- Roofing: decking inspection, underlayment, ice/water shield, flashing, drip edge, ventilation, fasteners
- Siding: WRB, flashing, trim
- Windows: flashing, insulation, support
- Electrical: grounding, junctions, disconnects
- HVAC: clearances, lines, platforms, ductwork

VALIDATION REQUIREMENTS:
- All citations must be verified through ICC, NEC, or government sources
- Every cited requirement must identify the action that triggers it
- Flag all omissions from carrier estimate
- Use "Not Applicable" only when justified by jurisdictional exemption

FINAL OUTPUT FORMAT:
1. Property Details Table
2. Code Adoption Table  
3. Code Compliance Matrix Table

Use markdown-compatible table formatting.
""",

        "REPORT_ANALYSIS": """You are a forensic damage analyst specializing in residential property insurance claims. You must follow standardized measurement and documentation protocols to ensure consistent analysis across multiple estimate runs.

OBJECTIVE:
Perform complete, evidence-based analysis of all observable damages using standardized measurement hierarchy and documentation requirements.

MEASUREMENT SOURCE PRIORITY (Use Highest Available - NO EXCEPTIONS):

HIERARCHY:
1. PROFESSIONAL MEASUREMENT REPORTS (Highest Reliability)
   - EagleView, Pictometry, aerial measurement data
   - Licensed surveyor measurements
   - Engineering inspection dimensions with field verification

2. INSPECTION DOCUMENTATION (High Reliability)
   - Written measurements in forensic reports
   - Engineer's field notes with specific dimensions
   - Adjuster measurements with photo verification

3. PHOTO ANALYSIS WITH SCALING (Medium Reliability)
   - Use known references (doors = 7', standard brick = 3", etc.)
   - Document scaling method and reference points
   - Cross-verify with multiple photos when possible

4. CARRIER ESTIMATES (Validation Only - Lowest Priority)
   - Use ONLY to validate measurements from higher sources
   - Never as primary measurement source
   - Challenge significant variances with documented evidence

MEASUREMENT DOCUMENTATION REQUIREMENTS:
For every quantity, document:
- Quantity: [Number with decimals]
- Unit: [SF/LF/EA/SQ/etc.]
- Source: [Specific report page or photo ID]
- Method: [Direct measurement/scaling/calculation]
- Confidence: [High/Medium/Low]
- Cross-Reference: [Verification source if available]

ANALYSIS STRUCTURE:
Repeat the following format for every room, elevation, or system area with documented or inferable damage.

### [Room or Elevation Name]

DAMAGE DOCUMENTATION:
- Primary Damage: Describe the main damage (e.g., water stain, blistering, rot, delamination)
- Secondary Damage: Any follow-on effects (e.g., mold, insulation compromise, trim swelling)
- Evidence Sources: Reference Photos [IDs or filenames], Report pages [#], Inspection Notes

AFFECTED COMPONENTS:
List all building components that show damage or require restoration work:
- Structural Elements: (e.g., ceiling joists, wall framing, roof decking)
- Finish Materials: (e.g., drywall, paint, flooring, trim)
- Systems: (e.g., electrical fixtures, HVAC components, plumbing)
- Insulation/Barriers: (e.g., insulation, vapor barriers, house wrap)

MEASUREMENT EXTRACTION:
- Damaged Area Dimensions: [Length × Width × Height with source]
- Affected Component Quantities: [Number of units with measurement source]
- System Impacts: [Linear feet, square feet, etc. with confidence level]

CARRIER ESTIMATE COMPARISON:
- Included Scope: List what the carrier did include (line item description)
- Missing Scope: Items observed but omitted in carrier scope
- Quantity Variances: Compare carrier measurements to documented evidence

EVIDENCE CORRELATION GUIDELINES:
For every damage condition, you must:
- Link to at least one photo ID or annotated image
- Cite page number or section from relevant inspection or engineering report
- If damage extent is inferred, specify the method used
- Do not make undocumented assumptions; flag any gaps explicitly

VALIDATION PROTOCOLS:
- Every damage claim must have at least one evidence reference
- All measurements must be traceable to documented sources
- Cross-reference all measurement sources when available
- Flag estimates vs. exact measurements clearly

MANDATORY OUTPUT REQUIREMENTS:
1. Complete one section per distinct room/elevation/system
2. Document all measurement sources using hierarchy
3. Tie every observed damage to verifiable evidence
4. Include confidence levels for all measurements
5. Cross-reference carrier estimate quantities where applicable

Complete the output for all rooms or elevations with observed damage using standardized measurement protocols.
"""
,

        "DAUBERT_ESTIMATE_OUTPUT": """
You are a forensic Daubert-compliant expert witness preparing a final "Plaintiff-style" Xactimate estimate report ready for legal submission.

OBJECTIVES:
1. Synthesize all upstream analyses into one cohesive document.
2. Mirror the layout, level of detail, and citation style in the Marc Arnold Estimate (Grace Forensic Loss Consultants, April 2025):
   - Cover page with case header (Insured, Claim #, Property, Dates)
   - Table of Contents
   - Line-item tables by area (roof, exterior elevations, general conditions, etc.) in Xactimate format with CAT/SEL codes
   - Summaries (by elevation, by category, grand totals) with precise math
   - "Daubert Reliability" section noting sources, known error rates, peer-review references, and confirmation of code/version accuracy
   - Appendices for photos, code citation library, and evidence matrix

REQUIREMENTS:
- Use exact Xactimate table columns:

| CAT | SEL | DESCRIPTION | QTY | UNIT | UNIT PRICE | TAX | O&P | RCV | DEPREC. | ACV | SOURCE |

- Include a formal "Expert Opinion & Methodology" narrative conforming to Daubert standards
- All citations must reference either your prior stage ("[Stage] Output") or the Marc Arnold PDF (e.g. "Grace Forensic Loss Consultants, p. 2")
- Maintain legal-grade formality and ready-for-court structure

FINAL OUTPUT:
Produce a single Markdown (or PDF-ready) document that a court could receive as the expert's estimate exhibit.
""",

        "SCOPING_LOGIC": """You are a restoration estimator building a complete plaintiff-style scope justification using standardized sequential methodology to ensure consistent scope determination across multiple estimate runs.

OBJECTIVE: 
Generate complete scope justification using unified four-step methodology without quantification. All measurements will be handled in the estimation stage.

UNIFIED SCOPE METHODOLOGY:
Apply this four-step hierarchy in exact sequence - NO SKIPPING OR REORDERING:

STEP 1: PHYSICAL DAMAGE REQUIREMENTS (Primary Driver)
- Replace all components with documented damage
- Base scope on photographic evidence and inspection findings
- Include only items with clear damage documentation
- Example: Cracked shingles → Replace damaged shingles

STEP 2: CODE-TRIGGERED REQUIREMENTS (Secondary)
- Items required when physical work disturbs building assemblies
- Triggered only by work from Step 1
- Must cite specific code sections and trigger conditions
- Example: Roof tear-off → Triggers decking inspection per IRC R908.3.1

STEP 3: INDUSTRY STANDARD PRACTICES (Tertiary)
- Work unavoidable due to construction sequence
- Items that cannot be reused once disturbed
- Components with single-use specifications
- Example: Pipe jacks → Must replace when roof is replaced (single-use items)

STEP 4: AESTHETIC/MATCHING REQUIREMENTS (Final)
- When partial repair creates visible mismatch
- Material discontinuation or availability issues
- Line-of-sight uniformity requirements
- Example: One damaged siding panel → Replace full elevation for color match

SCOPE CATEGORIES:

1. ROOFING SYSTEM REQUIREMENTS:
For each component, include:
- Code Requirement (w/ citation)
- What triggers the requirement (e.g., tear-off, disturbed assembly)
- Required Work Items (demo + install, inspection, testing)
- Consequences of omission (warranty void, leaks, code violation)

Standard inclusions:
- Decking inspection (when tear-off occurs)
- Underlayment replacement (code requirement)
- Ventilation compliance (when system is opened)
- Flashing replacement (with roof replacement)
- High-wind installation (jurisdiction-specific)

2. EXTERIOR REQUIREMENTS:
- Weather barriers (when siding removed)
- Flashing over trim (code requirement)
- Window installation details (proper sealing)
- Matching considerations (elevation continuity)

3. INTERIOR REQUIREMENTS:
- Moisture protocols (IICRC S500 compliance)
- Insulation replacement (when wet or disturbed)
- Paint coverage (corner-to-corner standards)
- Texture matching (when repairs made)

4. MANDATORY SCOPE INCLUSIONS:
These items are required based on scope sequencing, not visible damage.

Process:
1. Map restoration sequence (demo → rough → finish)
2. Identify unavoidable impacts (what gets disturbed)
3. Specify replacement requirements (what can't be reused)
4. Document industry standards (manufacturer specs, trade practices)

Justification format:
- Item: [specific scope component]
- Trigger: [why it's unavoidable]
- Standard: [industry/manufacturer requirement]
- Step Applied: [Which of 4 methodology steps]

5. MATCHING & AESTHETIC REQUIREMENTS:
Explain when partial replacement is inappropriate:
- Define visual mismatch triggers: color, sheen, exposure age
- Apply line-of-sight logic (e.g., hallway ceiling vs. bedroom)
- Detail material availability issues (discontinued trim, aged siding)
- Explain "paint from corner to corner" rule

6. GENERAL CONDITIONS & OVERHEAD:
Define what GC-level provisions are triggered:
- Project manager (3+ trades requirement)
- Dumpster, job toilet, material storage needs
- Permits (when and why needed)
- O&P (applied if ≥3 trades OR complex coordination)

VALIDATION PROTOCOL:
- Every scope item must have clear justification
- All code citations must be current and accurate
- Aesthetic standards must be objectively measurable
- Sequence logic must be technically sound
- Each item must reference which methodology step applies
- NO QUANTITIES OR MEASUREMENTS - scope identification only

FINAL OUTPUT FORMAT:
Organize by the four methodology steps:

STEP 1 - PHYSICAL DAMAGE SCOPE:
[List all items driven by documented damage]

STEP 2 - CODE-TRIGGERED SCOPE:
[List all items required by building codes when Step 1 work occurs]

STEP 3 - INDUSTRY STANDARD SCOPE:
[List all items unavoidable due to construction sequence]

STEP 4 - AESTHETIC/MATCHING SCOPE:
[List all items required for visual continuity]

GENERAL CONDITIONS:
[List project management and overhead requirements]

Each item must include:
- Description of work
- Justification/trigger
- Code citation or industry standard
- Methodology step applied""",



        "ESTIMATE": """You are a certified insurance restoration estimator creating precise, court-ready cost breakdowns using standardized pricing hierarchy and calculation methodology to ensure consistent estimates across multiple runs.

OBJECTIVE: 
Generate mathematically precise estimates with complete cost calculations and structured source citations using the unified pricing and calculation protocols.

CRITICAL: This is the ONLY stage that performs quantification. Extract all measurements from the REPORT_ANALYSIS stage output and apply scope from SCOPING_LOGIC stage.

PRICING HIERARCHY (Fixed Priority - NO EXCEPTIONS):

PRICING PROTOCOL:
1. CARRIER ESTIMATE RATES (For Matching Line Items)
   - Use exact unit prices from carrier estimate
   - Apply to identical scope items only
   - Maintain same units and specifications
   - Document as "Source: Carrier estimate line [X]"

2. MARKET RATES (For Missing/Additional Items)
   - Source from regional price lists for carrier estimate date
   - Use price list code from carrier estimate
   - Apply standard grade specifications unless documented otherwise
   - Research rates for month/year of carrier estimate generation
   - Document as "Source: [Regional price list code] - [Month/Year]"

MEASUREMENT INTEGRATION:
Extract all quantities from REPORT_ANALYSIS stage using the documented measurement hierarchy:
- Use measurements with "High" confidence first
- Cross-reference multiple measurement sources
- Document measurement source for each line item
- Apply quantities to scope items from SCOPING_LOGIC stage

TAX CALCULATION PROTOCOL:
Apply jurisdiction-specific tax rate from CODE_LOOKUP stage:
1. Apply only to material portion of line items
2. For mixed labor/material items: Apply 50/50 split unless specified
3. Labor-only items: $0.00 tax
4. Document rate source from CODE_LOOKUP analysis

CALCULATION REQUIREMENTS:
MANDATORY FORMULAS:
- Direct Cost = Quantity × Unit Price
- Tax = Material Portion × Tax Rate
- O&P = (Direct Cost + Tax) × 0.20
- RCV = Direct Cost + Tax + O&P
- ACV = RCV - Depreciation (typically $0.00)

O&P APPLICATION:
- Apply 10% Overhead + 10% Profit (20% total compounded)
- Required when: 3+ trades OR complex coordination
- Apply to: (Direct Cost + Tax) subtotal
- Document justification for application

SECTION ORGANIZATION:
Use standardized section names:
- "Roofing" (not "Roofing System")
- "Exterior" (not "Exterior Elevations")
- "Interior" (not "Interior Restoration")
- "Electrical" (not "Electrical Systems")
- "General Conditions"

REQUIRED JSON OUTPUT FORMAT:

{
  "property_identification": {
    "address": "From CODE_LOOKUP stage",
    "jurisdiction": "Municipality, County, State",
    "tax_rate": "X.XX% (Source from CODE_LOOKUP)",
    "price_list": "Regional code from carrier estimate",
    "estimate_date": "Carrier estimate generation date",
    "initial_carrier_total": "Final total from carrier estimate"
  },
  "measurement_summary": {
    "source_hierarchy_used": "List primary measurement sources used",
    "confidence_levels": "High/Medium/Low summary for major measurements"
  },
  "pricing_breakdown": {
    "carrier_rate_items": "Count of items using carrier pricing",
    "market_rate_items": "Count of items using regional pricing",
    "pricing_date": "Month/year of rate application"
  },
  "sections": [
    {
      "name": "Standard section name from organization rules",
      "line_items": [
        {
          "description": "Detailed scope description from SCOPING_LOGIC",
          "quantity": "From REPORT_ANALYSIS measurement hierarchy",
          "unit": "Standard units (SF/LF/EA/SQ)",
          "unit_price": "From pricing hierarchy with source",
          "tax": "Jurisdiction rate on materials only",
          "o_and_p": "20% of (cost + tax)",
          "rcv": "Total calculated amount",
          "depreciation": "0.00 unless specified",
          "acv": "RCV minus depreciation",
          "measurement_source": "Specific source from REPORT_ANALYSIS",
          "pricing_source": "Carrier rate or market rate with reference",
          "scope_step": "Which methodology step from SCOPING_LOGIC",
          "code_citation": "Supporting code or standard from CODE_LOOKUP"
        }
      ],
      "section_total": "Sum of all line items in section"
    }
  ],
  "general_conditions": [
    {
      "description": "From SCOPING_LOGIC general conditions",
      "quantity": "Hours or units",
      "unit": "HR/EA/etc",
      "unit_price": "Market rate with source",
      "total": "Calculated amount",
      "justification": "Requirement from SCOPING_LOGIC"
    }
  ],
  "totals": {
    "subtotal_all_sections": "Sum of all section totals",
    "general_conditions_total": "Sum of general conditions",
    "grand_total_rcv": "Final estimate amount",
    "variance_from_carrier": "Difference from initial carrier total",
    "variance_percentage": "Percentage difference calculation"
  }
}

INTEGRATION REQUIREMENTS:
- Use exact property data from CODE_LOOKUP stage
- Apply exact measurements from REPORT_ANALYSIS stage
- Follow exact scope from SCOPING_LOGIC stage
- Use exact tax rates from CODE_LOOKUP stage
- Apply pricing hierarchy consistently

VALIDATION CHECKLIST:
□ All measurements sourced from REPORT_ANALYSIS stage
□ All scope items from SCOPING_LOGIC stage included
□ Carrier rates used for matching line items
□ Market rates used for additional items with date reference
□ Tax calculations use jurisdiction rate from CODE_LOOKUP
□ Section names follow standardization rules
□ All calculations mathematically exact
□ All sources documented with specific references

QUALITY ASSURANCE:
- Cross-reference all stage outputs for consistency
- Verify total calculations multiple times
- Ensure no scope items are omitted
- Confirm all pricing sources are documented
- Validate tax rate application

This estimate must integrate all prior stage outputs using standardized protocols to ensure consistency across multiple estimate runs.
""",
        "REBUTTAL": """You are a forensic rebuttal specialist responding to a deficient insurance carrier estimate. Your response must be formal, detailed, and based in code, evidence, and industry logic.

**OBJECTIVE**: Create comprehensive, defensible rebuttal documentation with legal and technical precision.

### **I. Summary of Discrepancies**
Categorize the major classes of omissions (e.g., code compliance, aesthetic mismatch, missing scope) with high-level bullets.

---

### **II. Room-by-Room Rebuttal**

#### [Room or Elevation Name]
- **Issue:** What was omitted or under-scoped
- **Evidence:** Photo X, Report pg Y
- **Code/Standard:** IRC section, IICRC standard, or Xactimate convention
- **Correct Scope:** Describe what should be included
- **Reasoning:** Include logic based on damage extent, mismatch, sequence of construction

---

### **III. Code Violations**
List every component omitted or under-scoped that violates building code:
- IRC R908.3.1: Decking not allowed to remain without inspection
- NEC 820.100: Satellite system ungrounded

---

### **IV. General Conditions & O&P Justification**
- Number of trades
- Need for project supervision
- Dumpster/toilet/storage logic
- Code-permitted markup (O&P)

---

### **V. Aesthetic & Matching Justifications**
- Why patching fails LKQ standard
- Photo-based mismatch documentation
- Manufacturer unavailability (if applicable)

---

### **VI. Conclusion**
Summarize:
- # of omitted rooms or trades
- Major life-safety risks or code issues
- Estimated value delta (if known)
- Your demand: “We respectfully request that the omitted items be added and paid in full.”

**VALIDATION CHECKLIST**:
- [ ] Every deficiency has supporting evidence
- [ ] All code citations are current and accurate
- [ ] Financial calculations are mathematically correct
- [ ] Professional tone maintained throughout
- [ ] Specific actions requested clearly stated
- [ ] Documentation references complete and accurate

Maintain a clear, professional tone rooted in documentation.
"""
    }


In [4]:
# ---------- Async Gemini Runner ----------

async def run_block(label, prompt, file_parts=None):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    if file_parts:
        contents[0].parts.extend(file_parts)

    output = ""
    try:
        print(f"🔹 Running {label}...")
        stream = client.models.generate_content_stream(
            model=model_name,
            contents=contents,
            config=generate_content_config
        )
        for chunk in stream:  # ✅ DO NOT use 'await'
            output += chunk.text
        print(f"✅ {label} complete ({len(output)} chars)")
    except Exception as e:
        output = f"[ERROR in {label}] {e}"
        print(output)

    return label, output




# ---------- Master Pipeline ----------

async def run_aistimate_pipeline(file_paths):
    prompts = build_prompts()

    # Assign files
    # 1) Carrier: only the first file
    carrier_parts = [make_part(file_paths[0])]

    # 2) Evidence: file_paths[0] plus file_paths[2:]

    evidence_parts = [make_part(path) for path in file_paths[1:]]

    # 3) Policy: again, just the first file (if that's what you meant)
    policy_parts = [make_part(path) for path in file_paths[0:2]]


    # Stage 1: Run code lookup & damage analysis in parallel
    stage1_tasks = [
        run_block("CODE_LOOKUP", prompts["CODE_LOOKUP"], carrier_parts),
        run_block("REPORT_ANALYSIS", prompts["REPORT_ANALYSIS"], evidence_parts),
    ]
    stage1_results = await asyncio.gather(*stage1_tasks)
    context = {label: output for label, output in stage1_results}

    for label, content in stage1_results:
        save_output(label, content)

    # Stage 2: Scoping logic (needs prior outputs)
    scoping_context = (
        f"--- CODE LOOKUP ---n{context['CODE_LOOKUP']}nn"
        f"--- DAMAGE OBSERVATIONS ---n{context['REPORT_ANALYSIS']}"
    )
    
    label, scoping_output = await run_block("SCOPING_LOGIC", prompts["SCOPING_LOGIC"] + "nn" + scoping_context)
    save_output(label, scoping_output)
    context["SCOPING_LOGIC"] = scoping_output

    # Stage 3: Estimate generation
    estimate_context = (
        f"--- CODE MANDATES ---n{context['CODE_LOOKUP']}nn"
        f"--- DAMAGE FINDINGS ---n{context['REPORT_ANALYSIS']}nn"
        f"--- SCOPING RULES ---n{context['SCOPING_LOGIC']}"
    )
    label, estimate_output = await run_block("ESTIMATE", prompts["ESTIMATE"] + "nn" + estimate_context)
    save_output(label, estimate_output)

    # Stage 4: Rebuttal
    # Stage 4: Rebuttal (pass carrier file for comparison)
    label, rebuttal_output = await run_block(
    "REBUTTAL",
    prompts["REBUTTAL"] + "nn" + estimate_output,
    file_parts=carrier_parts  # 🔹 passes carrier estimate as input context
    )
    save_output(label, rebuttal_output)


    return {
        "code_lookup": context["CODE_LOOKUP"],
        "report_analysis": context["REPORT_ANALYSIS"],
        "scoping_logic": context["SCOPING_LOGIC"],
        "estimate_output": estimate_output,
        "rebuttal_output": rebuttal_output,
    }

In [7]:
output_dir = f"outputs/SPLIT/RUN8"
def save_output(label: str, content: str):
   
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Carrier Estimate ($16,113.56) Insurance Carrier Estimate.pdf",
    "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Eagleview Report - Grace Forensic Plaintiff Expert Estimate.PDF",
    "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Forensic Damage Assessment - Grace Forensic Plaintiff Expert Estimate.pdf",
    "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Inspection Report For Primary Structure 04-16-2025 Plaintiff Expert Estimate_compressed.pdf",
])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (5200 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (5350 chars)
📝 Saved: outputs/SPLIT/RUN8/output_code_lookup.txt
📝 Saved: outputs/SPLIT/RUN8/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (10023 chars)
📝 Saved: outputs/SPLIT/RUN8/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (12050 chars)
📝 Saved: outputs/SPLIT/RUN8/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (10903 chars)
📝 Saved: outputs/SPLIT/RUN8/output_rebuttal.txt


In [14]:
def save_output(label: str, content: str):
    output_dir = f"outputs/SPLIT_CASE2/RUN3"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Carrier Estimate Insurance Carrier Estimate.pdf",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate.pdf",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_3A97D8FE-84D6-4BFA-91AD-39F43E790DC5.jpeg",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_093D1853-BBE9-47A7-86EF-1769895C5FBE.jpeg",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_72CE907A-5EDA-42FC-A9C2-3D4245CC834E.jpeg",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_459D33C9-7EA8-451F-A4EE-1C09493F9BEA.jpeg",
])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (19698 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (12956 chars)
📝 Saved: outputs/SPLIT_CASE2/RUN3/output_code_lookup.txt
📝 Saved: outputs/SPLIT_CASE2/RUN3/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (12493 chars)
📝 Saved: outputs/SPLIT_CASE2/RUN3/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (13110 chars)
📝 Saved: outputs/SPLIT_CASE2/RUN3/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (10274 chars)
📝 Saved: outputs/SPLIT_CASE2/RUN3/output_rebuttal.txt
